# 第 1 课：LLM 是什么

> 学完这节课，你会理解：Token、Prompt、Context Window、Temperature 这些基本概念。

---

## 1.1 从你熟悉的东西说起

你做前端，肯定调过 API：

```javascript
const response = await fetch('/api/users', {
  method: 'POST',
  body: JSON.stringify({ name: '小明' })
});
const data = await response.json();
```

LLM（Large Language Model，大语言模型）也是一个 API。只不过：

| 普通 API | LLM API |
|---------|--------|
| 输入：结构化数据（JSON） | 输入：自然语言文本 |
| 输出：结构化数据（JSON） | 输出：自然语言文本（也可以要求输出 JSON） |
| 逻辑：代码写死的 if/else | 逻辑：模型"理解"后生成 |
| 确定性：同样输入，同样输出 | 随机性：同样输入，每次输出可能不同 |

常见的 LLM：
- **GPT-4o / GPT-4.1**（OpenAI）
- **Claude 4 Opus / Sonnet**（Anthropic）
- **Gemini**（Google）
- **DeepSeek**（深度求索）
- **Qwen**（通义千问，阿里）

## 1.2 Token：LLM 的"字节"

你知道计算机存储用字节（byte），LLM 处理文本用 **Token**。

Token 不是"字"，也不是"词"，而是模型自己定义的文本片段：

```
英文："Hello world" → ["Hello", " world"] → 2 tokens
中文："你好世界"   → ["你好", "世界"]     → 2 tokens（大约）
代码："console.log" → ["console", ".", "log"] → 3 tokens
```

**粗略估算**：1 个中文字 ≈ 1~2 tokens，1 个英文单词 ≈ 1 token。

### 为什么要关心 Token？

两个原因：

1. **花钱**：API 按 token 计费。GPT-4o 大约 $2.5 / 百万输入 token
2. **有上限**：每次调用能处理的 token 数量有限（Context Window），超过了就"装不下"

下面实际数一下 token。

In [ ]:
# 先安装 tiktoken（OpenAI 的 token 计算库）
# 如果已安装可以跳过
!pip install tiktoken -q

In [ ]:
import tiktoken

# GPT-4o 使用的编码器
enc = tiktoken.encoding_for_model("gpt-4o")

# 试试不同的文本
texts = [
    "Hello world",
    "你好世界",
    "他喜欢打羽毛球，每周三晚上都会去",
    "console.log('Hello')",
]

for text in texts:
    tokens = enc.encode(text)
    print(f"\n文本: {text}")
    print(f"Token 数量: {len(tokens)}")
    # 把每个 token 解码回文字看看
    print(f"Token 拆分: {[enc.decode([t]) for t in tokens]}")

### 练习：算算你们记忆系统的 token 消耗

假设一个联系人有：
- 关系小结：200 字
- 社交 Tips：300 字
- 最近 10 条对话事实，每条 50 字

把这些都注入 prompt，大概多少 token？

In [ ]:
# 试着算一下
sample_memory = """
【关系小结】
小明是你的大学同学，毕业后在同一个城市工作。他性格开朗，喜欢运动，
尤其是羽毛球和跑步。你们每周会约一次饭，聊聊工作和生活。
他最近在考虑跳槽，对职业发展比较焦虑。

【社交 Tips】
1. 小明比较在意朋友的主动关心，如果你很久没联系他，他会有点失落
2. 聊天时避免提到他的前女友（去年分手，还没完全走出来）
3. 他喜欢被夸奖，尤其是工作上的成就
4. 约饭的话他偏好川菜和日料，不太能吃辣但喜欢尝试
5. 他最近在学吉他，可以聊聊音乐话题

【最近对话】
- 小明说：周末去爬山吗？
- 我说：好啊，去哪个山？
- 小明说：香山怎么样，秋天叶子应该红了
- 我说：行，周六早上出发？
- 小明说：好的，我开车
"""

tokens = enc.encode(sample_memory)
print(f"这段记忆的 token 数: {len(tokens)}")
print(f"\n对比：GPT-4o 的 context window 是 128,000 tokens")
print(f"这段记忆占 context window 的: {len(tokens)/128000*100:.2f}%")

## 1.3 Prompt：你发给 AI 的"请求"

Prompt 就是你发给 LLM 的文本输入。但和普通 API 不同，LLM 的对话是有**角色**的：

```python
messages = [
    {"role": "system", "content": "你是一个社交助手"},   # System Prompt
    {"role": "user", "content": "帮我回复小明"},         # 用户消息
    {"role": "assistant", "content": "好的，..."},       # AI 的回复
    {"role": "user", "content": "换个语气"},             # 用户继续说
]
```

### 三种角色

| 角色 | 作用 | 前端类比 |
|-----|------|--------|
| **system** | 设定 AI 的身份和行为规则 | 组件的 props.config |
| **user** | 用户说的话 | 用户的输入事件 |
| **assistant** | AI 的回复 | 组件的输出/渲染 |

### System Prompt 的重要性

System Prompt 是整个 AI 应用的"灵魂"。同一个 LLM，不同的 System Prompt 会表现得完全不同：

```python
# System Prompt 1：通用助手
"你是一个有用的助手"

# System Prompt 2：你们的键盘回复助手
"你是一个帮用户生成聊天回复的助手。根据联系人信息和对话上下文，
生成自然、得体的回复建议。语气要匹配用户平时的说话风格。"

# System Prompt 3：你们的 OCR 识别
"你是一个聊天截图识别专家。分析截图中的聊天内容，
提取联系人昵称、平台类型、消息列表。以 JSON 格式输出。"
```

你们系统的记忆注入，本质上就是**把联系人信息塞到 System Prompt 里**。

## 1.4 Context Window：AI 的"工作内存"

这是理解记忆系统**为什么存在**的关键概念。

**Context Window** = LLM 一次能处理的最大 token 数。

| 模型 | Context Window |
|------|---------------|
| GPT-4o | 128K tokens |
| Claude Opus 4.6 | 200K tokens |
| Gemini 2.5 Pro | 1M tokens |

128K tokens 听起来很多？算一下：

- 一条微信消息平均 30 字 ≈ 40 tokens
- 128K tokens ≈ 3200 条消息
- 如果一个联系人每天聊 50 条，那只能装 **64 天**的聊天记录
- 而且你还有很多联系人！

**更重要的是**：塞太多内容会让 AI 性能下降。研究表明超过 32K tokens 后，AI 就开始"忘记"中间的内容（Lost in the Middle 效应）。

### 所以记忆系统要做的事情是：

```
海量聊天记录（几万条）
      ↓ 提取+压缩
关键信息摘要（几百字）
      ↓ 注入
Prompt（在 context window 里占很小一部分）
```

把"大海"压缩成"一杯水"，这就是记忆系统的核心价值。

## 1.5 Temperature：控制 AI 的"创造力"

调 LLM API 时有个参数叫 `temperature`（温度），控制输出的随机性：

| Temperature | 效果 | 适用场景 |
|------------|------|--------|
| 0 | 几乎确定性输出，每次结果相同 | 信息提取、分类、JSON 输出 |
| 0.3~0.7 | 有一定变化但基本靠谱 | 普通对话、回复生成 |
| 1.0 | 比较随机，更有"创意" | 创意写作、头脑风暴 |
| >1.0 | 非常随机，可能胡说八道 | 通常不用 |

你们系统中：
- **OCR 识别**用低温度（需要准确提取信息）
- **生成回复建议**用中等温度（需要自然但不跑偏）
- **生成关系小结**用中低温度（需要准确但有可读性）

## 1.6 本课小结

| 概念 | 一句话解释 | 你们系统中的对应 |
|------|----------|---------------|
| **Token** | LLM 处理文本的最小单位，1 中文字 ≈ 1-2 tokens | API 计费、context window 计算 |
| **Prompt** | 发给 LLM 的输入文本 | 键盘服务组装的请求 |
| **System Prompt** | 设定 AI 角色和行为的指令 | 记忆信息就是拼在这里面 |
| **Context Window** | LLM 一次能处理的最大 token 数 | 记忆系统存在的根本原因 |
| **Temperature** | 控制输出随机性 | OCR 用低温度，回复生成用中温度 |

### 下节预告

下一课我们会实际调 LLM API，亲手发一条消息，看看 AI 怎么回复。

## 1.7 延伸阅读

- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering) — 官方入门
- [通往 AGI 之路 - LLM 基础](https://waytoagi.feishu.cn/wiki/QPe5w5g7UisbEkkow8XcDmOpn8e) — 中文，概念梳理
- [吴恩达 Prompt Engineering 短课](https://www.deeplearning.ai/short-courses/chatgpt-prompt-engineering-for-developers/) — 免费视频课，1-2 小时